In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import platform
import seaborn as sns

import os
# OS별 폰트 설정
if os.name == "posix":  # macOS or Linux
    if platform.system() == "Darwin":  # macOS
        mpl.rc('font', family='AppleGothic')
    else:  # Linux
        # 보통 Ubuntu에 Noto Sans CJK 설치 권장
        mpl.rc('font', family='NanumGothic')  
elif os.name == "nt":  # Windows
    mpl.rc('font', family='Malgun Gothic')
    
# 마이너스 깨짐 방지
mpl.rcParams['axes.unicode_minus'] = False

---
```
spotify API 에서 가져올 수 있는 데이터
release_date
popularity
```
---

In [15]:
test = pd.read_csv("../data/spotify/20250925/KrWeekly.csv")

In [20]:
test.head(20)

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
0,1,spotify:track:7tI8dRuH2Yc6RuoTjxo4dU,Jimin,Who,BIGHIT MUSIC,1,1,62,1020755
1,2,spotify:track:3LWVXp636uLT356Rj08Jaz,Jimin,Be Mine,BIGHIT MUSIC,2,5,62,539309
2,3,spotify:track:59hVbgr8rfYkDbHfr8RcGI,Kenshi Yonezu,IRIS OUT,Sony Music Labels Inc.,3,23,2,491722
3,4,spotify:track:27xkOIER6uDLKALIelHylZ,Jin,Don’t Say You Love Me,BIGHIT MUSIC,2,2,19,443703
4,5,spotify:track:1CPZ5BxNNd0n0nF4Orb9JS,"HUNTR/X, EJAE, AUDREY NUNA, REI AMI, KPop Demo...",Golden,K-Pop Demon Hunters,1,3,14,364288
5,6,spotify:track:2HRgqmZQC0MC7GeNuDIXHN,"Jung Kook, Latto",Seven (feat. Latto) (Explicit Ver.),BIGHIT MUSIC,1,7,115,297798
6,7,spotify:track:7JQH2lBMZ10ztDnpoeq6Ki,데이먼스 이어 Damons year,yours,WM Korea,7,9,156,293356
7,8,spotify:track:6OWWZtNQORY1McaZmOrwhc,CORTIS,GO!,Republic Records – CORTIS,8,8,3,292718
8,9,spotify:track:1sUjTLLCJzcxC15GUBrOlg,aespa,Rich Man,SM Entertainment,4,4,3,275391
9,10,spotify:track:5H1sKFMzDeMtXwND3V6hRY,BLACKPINK,JUMP,YG Entertainment,3,6,11,266437


In [17]:
test.describe()

,rank,peak_rank,previous_rank,weeks_on_chart,streams
count,200.000000,200.000000,200.000000,200.000000,2.000000e+02
mean,100.500000,43.415000,90.550000,49.315000,1.143711e+05
std,57.879185,44.492705,59.369616,50.322535,9.419222e+04
min,1.000000,1.000000,-1.000000,1.000000,6.423000e+04
25%,50.750000,6.000000,38.750000,13.000000,7.264050e+04
50%,100.500000,25.500000,89.500000,31.000000,8.566550e+04
75%,150.250000,70.000000,141.250000,69.000000,1.215912e+05
max,200.000000,191.000000,199.000000,227.000000,1.020755e+06


In [18]:
test[test['weeks_on_chart'] == (test['weeks_on_chart'].max())]

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
142,143,spotify:track:5BqwC9kOBbqYkzdOKeXFFk,JANNABI,for lovers who hesitate,PEPONI MUSIC,55,144,227,74247


In [24]:
# 1. source별 등장 횟수 계산
counts = test['source'].value_counts()

# 2. 등장 횟수가 1인 source 값만 추출
unique_sources = counts[counts == 1].index

# 3. 원본 df에서 필터링
filtered = test[test['source'].isin(unique_sources)]
filtered

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
29,30,spotify:track:4xeugB5MqWh0jwvXZPxahq,WOODZ,Drowning,EDAM Entertainment,9,39,51,151260
31,32,spotify:track:77U0fCBBkM4MfTBUZBzaF9,"Mark Vank, Samu",4AM IN IBIZA,ICY,32,-1,1,142216
32,33,spotify:track:4LJJxB4a47mreJAfKwbGEM,DAYOUNG,body,STARSHIP Entertainment,33,164,2,141149
41,42,spotify:track:7wLJ4xzxNss5abZ1kXs242,"Car, the garden",Closely Far Away,니즈뮤직,38,50,54,129737
46,47,spotify:track:15HNdxGKNCIO9pgaY4n7FU,OFFICIAL HIGE DANDISM,Pretender,PONY CANYON INC.,47,56,67,124163
48,49,spotify:track:27x2IrIGwr56QWkqJ4cu9I,IU,"Bye, Summer",EDAM ENTERTAINMENT,30,30,3,123011
53,54,spotify:track:5BZsQlgw21vDOAjoqkNgKb,Justin Bieber,DAISIES,"JRC Entertainment, LLC / ILH Production Co. LL...",17,41,11,113250
54,55,spotify:track:2Gs0iF27my40p0dANv2rAg,"MXZI, Dj Samir, DJ Javi26",MONTAGEM XONADA,Lunar Media,55,73,5,111110
56,57,spotify:track:7kjVCymE7vn0HizmGCGFWQ,Lim Young Woong,Heavenly Ever After,Mulgogi Music / ADA,9,52,24,108858
62,63,spotify:track:7so0lgd0zP2Sbgs2d7a1SZ,"Lady Gaga, Bruno Mars",Die With A Smile,Interscope Records,6,63,58,106685


In [26]:
# 신규 진입 차트
# previous_rank 가 -1 (전주에 차트에 없었음)  +( peak_rank = rank ) 최고 랭크가 현재의 랭크임
test[(test['previous_rank'] == -1) & (test['rank'] == test['peak_rank'])]

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
31,32,spotify:track:77U0fCBBkM4MfTBUZBzaF9,"Mark Vank, Samu",4AM IN IBIZA,ICY,32,-1,1,142216
93,94,spotify:track:7zUD9iJoEOBRUE3UtHhdwR,PLAVE,숨바꼭질 (Hide and Seek),VLAST,94,-1,1,87187
135,136,spotify:track:4oE7MyJhqSD3BaHRpNs8Nl,"Kenshi Yonezu, Hikaru Utada",JANE DOE,Sony Music Labels Inc.,136,-1,1,75268
149,150,spotify:track:7MGUDGEQpcqf29gWAmJyy4,d4vd,Sleep Well,Darkroom/Interscope Records,150,-1,1,72697
190,191,spotify:track:5tEouf2s1SPwAIkOHnvWtQ,Beenzino,Aqua Man,Stone Music Entertainment,191,-1,5,65762


In [27]:
# 역주행 차트 
test[(test['previous_rank'] == -1) & (test['rank'] != test['peak_rank'])]

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams
76,77,spotify:track:7oYCBKvdjrqp5vDbhDBuac,Kenshi Yonezu,KICK BACK,Sony Music Labels Inc.,11,-1,66,98035
164,165,spotify:track:34HSEUn4YGAFBW9OHGIkU7,george,Boat,CRAFT AND JUN,56,-1,36,70058
171,172,spotify:track:3kXoKlD84c6OmIcOLfrfEs,"Earth, Wind & Fire",September,Columbia/Legacy,158,-1,2,69317
189,190,spotify:track:4PBtSN6dNFXA9RUkx5e3W9,Woody,Sadder Than Yesterday,GOLDEN MOON(GDM),129,-1,14,65895
191,192,spotify:track:0N8Xztr4pBHJ7V0moJWhbO,Aimyon,愛を伝えたいだとか,WM Japan,14,-1,42,65546
196,197,spotify:track:7JJmb5XwzOO8jgpou264Ml,Shawn Mendes,There's Nothing Holdin' Me Back,Island Records,108,-1,39,64457
199,200,spotify:track:2i3HoDE1ZvQ2bh2ZmgpSug,Nerd Connection,Silently Completely Eternally,유어썸머,85,-1,27,64230
